# TEST — Silver 2 Combine
Standalone test version of `silver2/combine.ipynb`.  
All parameters are hardcoded below — just edit and **Run All**.

In [ ]:
# ============================================================
# TEST PARAMETERS — edit these to match your test scenario
# ============================================================
entity       = "carrier_invoice"
source_names = ["fedex_direct_invoice", "dhl_direct_invoice", "ups_carrier_report"]

WORKSPACE   = "dev-fabric-data"
LAKEHOUSE   = "fabric_lakehouse"
DW_ENDPOINT = "8dac972c-0ff1-400f-bba3-e1e2302070b5.datawarehouse.fabric.microsoft.com"
DW_DATABASE = "fabric_gold_warehouse"

In [ ]:
import sys
from functools import reduce

from pyspark.sql import SparkSession, DataFrame
import pyspark.sql.functions as F

silver2_table = entity

### Shared lib & Config

In [ ]:
sys.path.insert(0, "/lakehouse/default/Files/shared")
import shared_functions as gf

run_id = "test-manual"

print(f"[TEST silver2/combine] entity={entity} sources={source_names}")

spark = SparkSession.builder.getOrCreate()

### Canonical schema & column mapping

In [ ]:
CANONICAL_COLUMNS = [
    "invoice_number",
    "invoice_date",
    "carrier",
    "purchase_contract",
    "tracking_number",
    "shipment_date",
    "origin_country",
    "destination_country",
    "weight_kg",
    "charge_amount",
    "charge_currency",
    "service_type",
    "account_number",
    "_source_name",
    "_source_file",
    "_run_id",
]

COLUMN_MAP: dict[str, dict[str, str]] = {
    "fedex": {
        "invoice_number":    "invoice_no",
        "invoice_date":      "invoice_date",
        "carrier":           None,
        "tracking_number":   "tracking_id",
        "shipment_date":     "ship_date",
        "weight_kg":         "weight_lbs",
        "charge_amount":     "net_charge",
        "charge_currency":   "currency",
        "service_type":      "service_code",
        "account_number":    "account_no",
    },
    "dhl": {
        "invoice_number":    "invoice_number",
        "invoice_date":      "billing_date",
        "carrier":           None,
        "tracking_number":   "waybill_no",
        "shipment_date":     "shipment_date",
        "weight_kg":         "weight_kg",
        "charge_amount":     "total_charge",
        "charge_currency":   "currency_code",
        "service_type":      "product_code",
        "account_number":    "shipper_account",
    },
    "ups": {
        "invoice_number":    "invoice_nbr",
        "invoice_date":      "invoice_dt",
        "carrier":           None,
        "tracking_number":   "pkg_tracking_no",
        "shipment_date":     "pickup_date",
        "weight_kg":         "billed_weight",
        "charge_amount":     "billed_amount",
        "charge_currency":   "currency",
        "service_type":      "service_description",
        "account_number":    "ups_account",
    },
}

LBS_TO_KG = 0.453592
all_frames = []

### 1. Read each staging table and normalise

In [ ]:
for sn in source_names:
    staging_table = f"staging_{sn}"
    staging_abfss = gf.onelake_abfss(WORKSPACE, LAKEHOUSE, f"Tables/silver/{staging_table}")

    print(f"  Reading {staging_table} ...")
    try:
        sdf = spark.read.format("delta").load(staging_abfss)
    except Exception as exc:
        raise RuntimeError(f"Cannot read {staging_table} — did silver1 succeed? {exc}") from exc

    carrier = sn.split("_")[0]
    mapping = COLUMN_MAP.get(carrier, {})

    select_exprs = []
    for canonical_col in CANONICAL_COLUMNS:
        if canonical_col == "carrier":
            select_exprs.append(F.lit(carrier).alias("carrier"))
        elif canonical_col == "purchase_contract":
            select_exprs.append(F.lit(sn).alias("purchase_contract"))
        elif canonical_col.startswith("_"):
            if canonical_col in sdf.columns:
                select_exprs.append(F.col(canonical_col))
            else:
                select_exprs.append(F.lit(None).cast("string").alias(canonical_col))
        else:
            src_col = mapping.get(canonical_col)
            if src_col and src_col in sdf.columns:
                if canonical_col == "weight_kg" and carrier == "fedex":
                    select_exprs.append(
                        (F.col(src_col).cast("double") * F.lit(LBS_TO_KG)).alias("weight_kg")
                    )
                else:
                    select_exprs.append(F.col(src_col).cast("string").alias(canonical_col))
            else:
                select_exprs.append(F.lit(None).cast("string").alias(canonical_col))

    normalised = sdf.select(select_exprs)
    all_frames.append(normalised)
    print(f"    {staging_table}: {normalised.count()} rows")

if not all_frames:
    raise RuntimeError("No staging frames loaded")

### 2. UNION ALL

In [ ]:
unified = reduce(DataFrame.unionAll, all_frames)
total_rows = unified.count()
print(f"  Unified rows: {total_rows}")

### 3. Write to Silver 2 Delta table

In [ ]:
silver2_abfss = gf.onelake_abfss(WORKSPACE, LAKEHOUSE, f"Tables/silver/{silver2_table}")

unified.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(silver2_abfss)
print(f"  Written to {silver2_abfss}")

### 4. Log

In [ ]:
with gf.dw_connection(DW_ENDPOINT, DW_DATABASE) as conn:
    gf.log_run(
        conn,
        source_name=entity,
        layer="silver2",
        status="success",
        rows_processed=total_rows,
        run_id=run_id,
    )

print(f"  Done: {total_rows} rows -> {silver2_table}")